# Notebook06 bounded mathematical study workflow
Definitions only. Run All in the main notebook starts a detached three-hour study including training-only attention audit, real-step calibration, exact recovery, paired training and fixed probes. One persisted clock and writer. Source/configuration changes require a new run ID; no silent restart. Expected elapsed experiment time: at most3hours. Artifacts: artifacts/notebook_06/<run_id>/. Full-state checkpoints: checkpoints/<run_id>/.

In [ ]:
"""Notebook06 orchestration: one persisted three-hour experimental clock."""
from pathlib import Path
import copy, hashlib, importlib.util, json, os, shlex, subprocess, sys, time
from datetime import datetime, timezone
import nbformat
from fmm_jepa.bootstrap import TaggedNotebookLoader
from fmm_jepa.training import atomic_json, digest_file, run_writer
MAIN_NOTEBOOK='notebooks/06_Bounded_Mathematical_Improvements.ipynb'
TERMINAL={'COMPLETED','FAILED','INCOMPLETE'}

def _read(path):return json.loads(Path(path).read_text())
def _normalized(value):return json.loads(json.dumps(value,allow_nan=False))
def _hash(value):return hashlib.sha256(json.dumps(_normalized(value),sort_keys=True,separators=(',',':')).encode()).hexdigest()
def _export_hash(path):
    return hashlib.sha256('\n'.join(''.join(c['source']) for c in _read(path)['cells'] if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])).encode()).hexdigest()
def _load(root,name):
    key='fmm06_'+name
    if key not in sys.modules:
        spec=importlib.util.spec_from_loader(key,TaggedNotebookLoader(Path(root)/'experiments/06_math'/(name+'.ipynb')))
        module=importlib.util.module_from_spec(spec);sys.modules[key]=module
        try:spec.loader.exec_module(module)
        except BaseException:sys.modules.pop(key,None);raise
    return sys.modules[key]

def source_hashes(root):
    return {str(p.relative_to(root)):_export_hash(p) for folder in ('fmm_jepa','experiments/04_followup','experiments/05_scaling','experiments/06_math') for p in sorted((Path(root)/folder).glob('*.ipynb'))}

def consumer_hash(root):return _hash([''.join(c['source']) for c in _read(Path(root)/MAIN_NOTEBOOK)['cells'] if c['cell_type']=='code'])

def study_configuration(run_id='math-improvements-001'):
    if not run_id or any(not(c.isalnum() or c in '-_') for c in run_id):raise ValueError('Safe nonempty run ID required')
    return {'schema':'fmm-jepa-notebook06-study/v1','run_id':run_id,'hard_limit_seconds':10800,
            'device':'cuda:0','dtype':'float32','image_size':128,'effective_batch':128,
            'seeds':[20261001,20261002,20261003],'variants':['vit','stable','centered_tree'],
            'diagnostic_seconds':600,'max_updates':600,'minimum_three_seed_updates':300,'minimum_two_seed_updates':200,
            'round_updates':50,'probe_epochs':20,'heldout_enabled':False,
            'budget_policy':'Prefer3pairedseeds at>=300updates; otherwise2 at>=200; reserve600seconds perseed for6fixed probes and300seconds for resume. Use1.30times calibration training cost.',
            'rank_policy':'Training-only saved-activation audit; per-layer16/32/48/64 with total128; retainuniform32 unless audit score improves by at least2percent.'}

def _status(artifact,stage,status,**details):
    result={'stage':stage,'status':status,'updated_unix':time.time(),**details}
    atomic_json(Path(artifact)/'study_status.json',result)
    print(stage+': '+status,flush=True)
    return result

def _validation(root):
    path=Path(root)/'artifacts/notebook_06_validation/operator_validation.json'
    report=_read(path)
    if report.get('status')!='PASS':raise ValueError('Notebook06 operator validation must pass before launch')
    if report.get('attention_export_sha256')!=_export_hash(Path(root)/'experiments/06_math/attention.ipynb'):
        raise ValueError('Operator validation does not match current attention definitions')
    return {'path':str(path.relative_to(root)),'sha256':digest_file(path),'attention_export_sha256':report['attention_export_sha256']}

def _check_binding(root,binding,config):
    if binding['configuration']!=_normalized(config):raise ValueError('Registered run configuration changed')
    if binding['source_hashes']!=source_hashes(root):raise ValueError('Registered source definitions changed; use a new run ID')
    if binding['consumer_notebook_sha256']!=consumer_hash(root):raise ValueError('Registered main notebook code changed')
    if binding['operator_validation']!=_validation(root):raise ValueError('Bound validation evidence changed')

def _clock(artifact,binding):
    result=_read(Path(artifact)/'budget_clock.json')
    if result!=binding['budget_clock'] or result['max_seconds']!=10800:raise ValueError('Original three-hour clock changed')
    return result

def _prepare(root,config):
    root=Path(root).resolve();artifact=root/'artifacts/notebook_06'/config['run_id'];artifact.mkdir(parents=True,exist_ok=True)
    path=artifact/'study_binding.json'
    if path.exists():
        binding=_read(path);_check_binding(root,binding,config)
    else:
        if (artifact/'budget_clock.json').exists() or (artifact/'stages').exists():raise ValueError('Unbound existing experiment evidence; refusing restart')
        validation=_validation(root);now=time.time()
        clock={'started_unix':now,'deadline_unix':now+10800,'max_seconds':10800,'policy':'Original launch clock includes audit, calibration, recovery verification, training and evaluation; never resets.'}
        binding={'schema':'notebook06-binding/v1','configuration':_normalized(config),'source_hashes':source_hashes(root),
                 'consumer_notebook_sha256':consumer_hash(root),'budget_clock':clock,'operator_validation':validation}
        atomic_json(path,binding);atomic_json(artifact/'budget_clock.json',clock)
    return artifact,binding,_clock(artifact,binding)

def _choose_budget(config,calibration,deadline):
    per_round=sum(calibration['selected_model_step_seconds'].values())
    if per_round<=0:raise ValueError('Invalid calibration time')
    remaining=deadline-time.time();choices=[]
    for count,minimum in ((3,config['minimum_three_seed_updates']),(2,config['minimum_two_seed_updates'])):
        reserve=600*count;preflight=300;fixed_overhead=180
        available=remaining-reserve-preflight-fixed_overhead
        updates=min(config['max_updates'],int(available/(1.30*count*per_round)//50)*50)
        choices.append({'seeds':count,'updates':max(0,updates),'minimum':minimum,'evaluation_reserve_seconds':reserve,
                        'predicted_training_seconds':1.30*count*per_round*max(0,updates)})
        if updates>=minimum:return {**choices[-1],'status':'PASS','remaining_seconds':remaining,'choices_considered':choices,'calibration_round_seconds':per_round}
    raise TimeoutError('Original three-hour budget cannot fit minimum paired comparison plus evaluation: '+str(choices))

def _plan(root,study,binding,bundle,base,calibration,feature_counts,budget):
    preparation=_load(root,'preparation');arms=[];seeds=study['seeds'][:budget['seeds']];updates=budget['updates']
    for index,seed in enumerate(seeds):
        for variant in study['variants']:
            cfg=preparation.arm_config(base,variant,feature_counts)
            cfg['training'].update(updates=updates,microbatch=calibration['selected_microbatch'],accumulation_steps=calibration['selected_accumulation_steps'])
            cfg['seeds'].update(model_init=seed,data_order=1101+10000*index,augmentation=2202+10000*index,mask=3303+10000*index,probe=4404)
            cfg['model']['feature_seed']=seed+1000
            cfg['evaluation'].update(batch_size=min(64,calibration['selected_microbatch']),diagnostic_steps=[0,updates])
            cfg['budget']['evaluation_reserve_seconds']=budget['evaluation_reserve_seconds']
            arms.append({'id':'s'+str(seed)+'-'+variant,'kind':'vit' if variant=='vit' else 'semantic','variant':variant,'seed':seed,'condition':variant,'config':cfg})
    return _normalized({'schema':'notebook06-stage-plan/v1','run_id':study['run_id'],'stage_id':'learning','stage_role':'learning',
         'stage':{'id':'learning','image_size':128,'pretrain_pool':'selected'},'updates':updates,'milestones':[updates],
         'round_updates':study['round_updates'],'effective_batch':128,'seeds':seeds,'variants':study['variants'],'arms':arms,
         'train_count':len(bundle['train_ids']),'probe_train_count':len(bundle['probe_train_ids']),
         'source_hashes':binding['source_hashes'],'consumer_notebook_sha256':binding['consumer_notebook_sha256'],
         'resume_notebook':MAIN_NOTEBOOK,'resume_cell':'launch-math-study','evaluation_reserve_seconds':budget['evaluation_reserve_seconds'],
         'study_configuration_hash':_hash(study),'numerical_backend':'math06-v1','heldout_status':'NOT_RUN',
         'selection':'Feature allocation from training-only approximation audit; schedule from actual timing before new quality outcomes. Fixed final probe epoch20.',
         'failure_policy':'Stop first numerical failure; never skip samples.','budget_selection':budget})

def run_study(root,run_id):
    root=Path(root).resolve();artifact=root/'artifacts/notebook_06'/run_id
    binding=_read(artifact/'study_binding.json');study=binding['configuration'];_check_binding(root,binding,study)
    clock=_clock(artifact,binding);deadline=clock['deadline_unix']
    if (artifact/'study_status.json').exists() and _read(artifact/'study_status.json').get('status')=='COMPLETED':return _read(artifact/'study_status.json')
    if time.time()>=deadline:raise TimeoutError('Original three-hour deadline expired')
    preparation,diagnostics,runner,evaluation=(_load(root,n) for n in ('preparation','diagnostics','runner','evaluation'))
    with run_writer(artifact):
        try:
            folder=artifact/'stages/learning';folder.mkdir(parents=True,exist_ok=True)
            (folder/'predictions').mkdir(exist_ok=True)
            base=preparation.base_config(root)
            _status(artifact,'verify_training_and_development_data','RUNNING')
            bundle=preparation.prepare_bundle(root,folder,base)
            _status(artifact,'saved_checkpoint_attention_audit','RUNNING')
            audit=diagnostics.audit(root,artifact,bundle=bundle,deadline_unix=deadline,device=study['device'],max_seconds=study['diagnostic_seconds'])
            atomic_json(artifact/'audit_result.json',audit)
            # The diagnostic owns its explicit recommendation field; never select on development accuracy.
            feature_counts=audit.get('recommended_feature_counts',audit.get('feature_counts'))
            if feature_counts is None:
                recommendation=audit.get('recommendation',{})
                feature_counts=recommendation.get('feature_counts',recommendation.get('per_layer_feature_counts'))
            if feature_counts is None:raise ValueError('Audit did not supply a registered rank recommendation')
            feature_counts=list(map(int,feature_counts))
            if len(feature_counts)!=4 or sum(feature_counts)!=128 or any(x not in (16,32,48,64) for x in feature_counts):raise ValueError('Invalid fixed-capacity allocation')
            _status(artifact,'actual_update_calibration','RUNNING',feature_counts=feature_counts)
            calibration=preparation.calibrate(root,folder,base,bundle,feature_counts,deadline)
            plan_path=folder/'plan.json'
            if plan_path.exists():
                plan=_read(plan_path)
                if plan['source_hashes']!=binding['source_hashes']:raise ValueError('Saved plan sources changed')
                if plan['study_configuration_hash']!=_hash(study):raise ValueError('Saved plan study changed')
                if any(a['config']['math06']['feature_counts']!=feature_counts for a in plan['arms'] if a['variant']=='centered_tree'):raise ValueError('Saved allocation changed')
            else:
                budget=_choose_budget(study,calibration,deadline)
                plan=_plan(root,study,binding,bundle,base,calibration,feature_counts,budget)
                preparation.immutable(plan_path,plan)
            _status(artifact,'exact_resume_preflight','RUNNING',updates=plan['updates'],seeds=plan['seeds'],variants=plan['variants'])
            runner.prepare_stage(root,folder,plan,bundle)
            preflight=runner.verify_stage_resume(root,folder,plan,bundle,deadline)
            atomic_json(folder/'preflight_result.json',preflight)
            if preflight.get('status')!='PASS':raise RuntimeError('Exact-resume preflight did not pass')
            _check_binding(root,binding,study)
            _status(artifact,'paired_training','RUNNING',updates=plan['updates'],seeds=plan['seeds'],variants=plan['variants'],feature_counts=feature_counts)
            training=runner.train_stage(root,folder,plan,bundle,deadline)
            atomic_json(folder/'training_result.json',training)
            _status(artifact,'frozen_feature_evaluation','RUNNING',updates=plan['updates'],seeds=plan['seeds'])
            result=evaluation.evaluate_stage(root,folder,plan,bundle,training,deadline)
            atomic_json(artifact/'study_results.json',result)
            if result.get('status') not in ('PASS','COMPLETE','COMPLETED'):raise TimeoutError('Evaluation incomplete under original deadline')
            _check_binding(root,binding,study)
            return _status(artifact,'complete','COMPLETED',elapsed_seconds=time.time()-clock['started_unix'],updates=plan['updates'],seeds=plan['seeds'],variants=plan['variants'],feature_counts=feature_counts,heldout_status='NOT_RUN',results=str(folder/'stage_results.md'))
        except BaseException as error:
            _status(artifact,'interrupted','INCOMPLETE' if isinstance(error,TimeoutError) else 'FAILED',error=type(error).__name__+': '+str(error),cause=repr(error.__cause__) if error.__cause__ else None,elapsed_seconds=time.time()-clock['started_unix'])
            raise

def _exists(session):return subprocess.run(['tmux','has-session','-t','='+session],capture_output=True).returncode==0

def launch_study(root,config):
    root=Path(root).resolve();artifact=root/'artifacts/notebook_06'/config['run_id']
    with run_writer(artifact/'launch_lock'):
        artifact,binding,clock=_prepare(root,config);session='fmm-math-'+config['run_id']
        if _exists(session):
            _load(root,'deadline_watchdog').launch_watchdog(root,config['run_id'])
            return {'status':'EXISTING_SESSION','session':session,'deadline_unix':clock['deadline_unix'],'artifact':str(artifact)}
        prior=_read(artifact/'study_status.json') if (artifact/'study_status.json').exists() else {}
        if prior.get('status')=='COMPLETED':return prior
        if time.time()>=clock['deadline_unix']:raise TimeoutError('Original deadline expired; Run All cannot reset it')
        setup=next(c.source for c in nbformat.read(root/MAIN_NOTEBOOK,as_version=4).cells if c.get('id')=='bootstrap-math-study')
        code=setup+'\nWORKFLOW.run_study(ROOT, '+repr(config['run_id'])+')\n'
        worker=artifact/'worker.ipynb'
        if worker.exists():
            if [c.source for c in nbformat.read(worker,as_version=4).cells if c.cell_type=='code']!=[code]:raise ValueError('Detached worker source changed')
        else:
            nbformat.write(nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# Notebook06 detached worker\nOutputs: this run folder; full-state checkpoints: checkpoints/'+config['run_id']+'/. Resume with unchanged Notebook06 Run All before the original three-hour deadline.'),nbformat.v4.new_code_cell(code)],metadata={'kernelspec':{'display_name':'Python (ahr-go)','language':'python','name':'ahr-go'}}),worker)
        command=[sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.timeout=-1','--ExecutePreprocessor.kernel_name=ahr-go',str(worker)]
        shell='exec '+shlex.join(command)+' > '+shlex.quote(str(artifact/'worker.log'))+' 2>&1'
        launch={'session':session,'artifact':str(artifact),'command':command,'attach':'tmux attach -t '+session,'detach':'Ctrl-b, then d','deadline_unix':clock['deadline_unix'],'deadline_utc':datetime.fromtimestamp(clock['deadline_unix'],timezone.utc).isoformat(),'binding_hash':_hash(binding)}
        atomic_json(artifact/'launch.json',launch);_status(artifact,'launching','STARTING')
        try:
            _load(root,'deadline_watchdog').launch_watchdog(root,config['run_id'])
            subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(root),'bash','-c',shell],check=True)
        except BaseException as error:_status(artifact,'launch_failed','FAILED',error=repr(error));raise
        return {'status':'LAUNCHED',**launch}

def inspect_study(root,run_id):
    artifact=Path(root)/'artifacts/notebook_06'/run_id
    if not (artifact/'study_status.json').exists():return {'status':'NOT_LAUNCHED'}
    status=_read(artifact/'study_status.json');clock=_read(artifact/'budget_clock.json')
    result={**status,'session':'fmm-math-'+run_id,'deadline_utc':datetime.fromtimestamp(clock['deadline_unix'],timezone.utc).isoformat(),'remaining_minutes':max(0,(clock['deadline_unix']-time.time())/60)}
    rows=[]
    for p in sorted((Path(root)/'checkpoints'/run_id/'stages/learning').glob('arms/*/manifest.json')):
        m=_read(p);rows.append({'arm':p.parent.name,'successful_updates':m.get('successful_updates',m.get('latest_step'))})
    result['saved_training_progress']=rows
    return result
